# 03. 64-Frame Resizing

This notebook resizes the cleaned `data/interpolated_json` pose files into fixed 64-frame sequences for Hu/TransRAC-style processing.


## 1. Import Libraries and Set Paths

This cell imports the required libraries and defines the input and output folders.

The input folder is `data/interpolated_json`.

The output folder is `data/resized_interpolated_json`.


In [ ]:
from pathlib import Path
import json
import numpy as np
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LOCAL_PROJECT_ROOT = Path(
    "/Users/jhonmichaelferrer/Documents/VSCode/Coreset/CoreSet-AI-Fitness-Tracker-ML-Pipeline"
)
if not (PROJECT_ROOT / "data").exists() and LOCAL_PROJECT_ROOT.exists():
    PROJECT_ROOT = LOCAL_PROJECT_ROOT

INPUT_BASE = PROJECT_ROOT / "data" / "interpolated_json"
OUTPUT_BASE = PROJECT_ROOT / "data" / "resized_interpolated_json"

CATEGORIES = ["bench_press", "pull_up", "push_up", "squat"]
DISPLAY_NAMES = {
    "bench_press": "Bench Press",
    "pull_up": "Pull Up",
    "push_up": "Push Up",
    "squat": "Squat",
}

TARGET_FRAMES = 64
EXPECTED_LANDMARKS = 33
FEATURES = 4

print("Project root:", PROJECT_ROOT)
print("Input folder exists:", INPUT_BASE.exists())
print("Output folder:", OUTPUT_BASE)

for category in CATEGORIES:
    category_path = INPUT_BASE / category
    files = sorted(category_path.glob("*.json"))
    print(
        f"{category}: "
        f"folder_exists={category_path.exists()} "
        f"json_files={len(files)}"
    )


## 2. Convert Each Landmark Into Numeric Values

This function converts each landmark into `[x, y, z, visibility]`.

It supports both dictionary landmarks and list landmarks.


In [ ]:
def landmark_to_vector(landmark):
    if landmark is None:
        return [np.nan, np.nan, np.nan, np.nan]

    if isinstance(landmark, dict):
        return [
            float(landmark.get("x", np.nan)),
            float(landmark.get("y", np.nan)),
            float(landmark.get("z", np.nan)),
            float(landmark.get("visibility", np.nan)),
        ]

    if isinstance(landmark, list):
        values = landmark[:FEATURES]
        values = values + [np.nan] * (FEATURES - len(values))
        return [float(value) if value is not None else np.nan for value in values]

    return [np.nan, np.nan, np.nan, np.nan]


## 3. Convert JSON Frames Into an Array

This function converts JSON frames into a NumPy array with shape `frames x 33 x 4`.

It also keeps the original frame indices for resampling.


In [ ]:
def frames_to_array(frames):
    arr = np.full(
        (len(frames), EXPECTED_LANDMARKS, FEATURES),
        np.nan,
        dtype=np.float64,
    )
    frame_indices = []

    for frame_pos, frame in enumerate(frames):
        frame_indices.append(frame.get("frame_index", frame_pos))
        landmarks = frame.get("landmarks")

        if not isinstance(landmarks, list):
            continue

        for landmark_idx in range(min(EXPECTED_LANDMARKS, len(landmarks))):
            arr[frame_pos, landmark_idx] = landmark_to_vector(landmarks[landmark_idx])

    return arr, np.asarray(frame_indices, dtype=np.float64)


## 4. Fill Missing Landmark Values

This function fills missing values before temporal resizing.

It interpolates when possible, repeats a single valid value when needed, and uses `0.0` only as a final fallback.


In [ ]:
def fill_missing_flat(flat):
    filled = flat.copy()
    source_x = np.arange(flat.shape[0], dtype=np.float64)

    for col in range(flat.shape[1]):
        values = filled[:, col]
        valid = np.isfinite(values)

        if valid.sum() >= 2:
            filled[:, col] = np.interp(source_x, source_x[valid], values[valid])
        elif valid.sum() == 1:
            filled[:, col] = values[valid][0]
        else:
            filled[:, col] = 0.0

    return filled


## 5. Resize Each Pose Sequence to 64 Frames

This function resamples the full motion sequence into exactly 64 frames.

Longer files are compressed to 64 frames and shorter files are expanded to 64 frames using normalized-time interpolation.


In [ ]:
def temporal_resample_array(arr, target_frames=TARGET_FRAMES):
    source_frames = arr.shape[0]

    if source_frames == 0:
        raise ValueError("No frames found")

    flat = arr.reshape(source_frames, -1)
    flat = fill_missing_flat(flat)

    if source_frames == target_frames:
        resized = flat
    elif source_frames == 1:
        resized = np.repeat(flat, target_frames, axis=0)
    else:
        source_t = np.linspace(0.0, 1.0, source_frames)
        target_t = np.linspace(0.0, 1.0, target_frames)
        resized = np.empty((target_frames, flat.shape[1]), dtype=np.float64)

        for col in range(flat.shape[1]):
            resized[:, col] = np.interp(target_t, source_t, flat[:, col])

    resized = resized.reshape(target_frames, EXPECTED_LANDMARKS, FEATURES)
    resized[:, :, 3] = np.clip(resized[:, :, 3], 0.0, 1.0)

    return resized


## 6. Resize Frame Indices

This function resamples the original frame indices so each 64-frame output still references approximate source-frame positions.


In [ ]:
def temporal_resample_frame_indices(frame_indices, target_frames=TARGET_FRAMES):
    if len(frame_indices) == 0:
        return list(range(target_frames))

    if len(frame_indices) == 1:
        return [int(frame_indices[0])] * target_frames

    source_t = np.linspace(0.0, 1.0, len(frame_indices))
    target_t = np.linspace(0.0, 1.0, target_frames)

    return [int(round(value)) for value in np.interp(target_t, source_t, frame_indices)]


## 7. Resize One JSON File

This function reads one interpolated JSON file, resizes it to 64 frames, and writes the result to `data/resized_interpolated_json`.

The output keeps metadata such as `video_path` and `fps`, then adds `original_frame_count`, `resized_frame_count`, and `resize_method`.


In [ ]:
def resize_file(json_path):
    relative_path = json_path.relative_to(INPUT_BASE)
    output_path = OUTPUT_BASE / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with json_path.open("r") as f:
        data = json.load(f)

    frames = data.get("frames", [])
    arr, frame_indices = frames_to_array(frames)
    resized_arr = temporal_resample_array(arr)
    resized_frame_indices = temporal_resample_frame_indices(frame_indices)

    output_data = {key: value for key, value in data.items() if key != "frames"}
    output_data["original_frame_count"] = len(frames)
    output_data["resized_frame_count"] = TARGET_FRAMES
    output_data["resize_method"] = "temporal_resampling_linear_after_cubic_spline"
    output_data["frames"] = [
        {
            "frame_index": resized_frame_indices[frame_idx],
            "landmarks": resized_arr[frame_idx].tolist(),
        }
        for frame_idx in range(TARGET_FRAMES)
    ]

    with output_path.open("w") as f:
        json.dump(output_data, f, separators=(",", ":"))

    return relative_path.parts[0]


## 8. Run 64-Frame Resizing for All Categories

This cell processes all JSON files inside `data/interpolated_json` and saves the resized versions in `data/resized_interpolated_json`.


In [ ]:
json_paths = []

for category in CATEGORIES:
    category_files = sorted((INPUT_BASE / category).glob("*.json"))
    json_paths.extend(category_files)

print("Total input JSON files found:", len(json_paths))

if not json_paths:
    raise FileNotFoundError(f"No JSON files found in {INPUT_BASE}")

category_counts = Counter()

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(resize_file, path) for path in json_paths]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="temporal resample to 64 frames",
    ):
        category_counts[future.result()] += 1

print("completed", sum(category_counts.values()))
print("category_counts", dict(category_counts))
print("output", OUTPUT_BASE)


## 9. Summarize Original Frame Counts Before Resizing

This cell reports the lowest, highest, and average frame count before resizing.


In [ ]:
BASE = OUTPUT_BASE
counts = defaultdict(list)
missing_original_frame_count = []
json_paths = sorted(BASE.rglob("*.json"))

for path in json_paths:
    with path.open("r") as f:
        data = json.load(f)

    category = path.relative_to(BASE).parts[0]
    original_count = data.get("original_frame_count")

    if isinstance(original_count, int):
        counts[category].append(original_count)
    else:
        missing_original_frame_count.append(str(path.relative_to(BASE)))

all_counts = [count for category_counts in counts.values() for count in category_counts]

print("=" * 78)
print("ORIGINAL FRAME COUNT SUMMARY BEFORE 64-FRAME RESIZING".center(78))
print("=" * 78)
print(f"{'Category':<20} | {'Files':<6} | {'Lowest':<8} | {'Highest':<8} | {'Average':<8}")
print("-" * 78)

for category in CATEGORIES:
    values = counts.get(category, [])
    display_name = DISPLAY_NAMES[category]

    if values:
        print(
            f"{display_name:<20} | "
            f"{len(values):<6} | "
            f"{min(values):<8} | "
            f"{max(values):<8} | "
            f"{sum(values) / len(values):<8.2f}"
        )
    else:
        print(f"{display_name:<20} | {0:<6} | {'-':<8} | {'-':<8} | {'-':<8}")

print("-" * 78)

if all_counts:
    print(
        f"{'TOTAL':<20} | "
        f"{len(all_counts):<6} | "
        f"{min(all_counts):<8} | "
        f"{max(all_counts):<8} | "
        f"{sum(all_counts) / len(all_counts):<8.2f}"
    )
else:
    print(f"{'TOTAL':<20} | {0:<6} | {'-':<8} | {'-':<8} | {'-':<8}")

print("=" * 78)
print(f"JSON files found: {len(json_paths)}")

if missing_original_frame_count:
    print(f"Files missing original_frame_count: {len(missing_original_frame_count)}")
